# V1DD 1196 CCM ETL

Convert local V1DD release files to CCM tables.

In [1]:
import gc
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.feather as feather
from deltalake import DeltaTable, write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import (
    cell_cell_connectivity_to_arrow,
    iter_cell_cell_connectivity,
    read_embedding_locations,
    write_models,
)
from connects_common_connectivity.io.arrow_utils import (
    attach_linkml_metadata,
    build_arrow_schema,
    build_cell_feature_matrix_schema,
)
from connects_common_connectivity.io.path_spec import (
    MODEL_TABLE_PATHS,
    WIDE_PAYLOAD_PATHS,
)
from connects_common_connectivity.models import (
    AlgorithmRun,
    AnatomicalOrientation,
    CellCellConnectivityLong,
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    Cluster,
    ClusterHierarchy,
    ClusterMembership,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    Default2DView,
    EmbeddingLocation,
    EmbeddingMethod,
    EmbeddingSpace,
    HierarchyCategory,
    LocationType,
    Modality,
    ReferenceSpace,
    SignedAxis,
    SpatialLocation,
    SynapseConnectivityLong,
    SynapseFeatureMatrix,
    Unit,
)

In [2]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/EM v1dd 1196"
COLORS_PATH = REPO_ROOT / "data/EM cell types/em_cell_type_colors_csm_202610.rtf"
OUTPUT_ROOT = Path(get_settings().output_root)

PROJECT_ID = "v1dd_1196"
DATASET_SINGLE_SOMA_NEURONS = "v1dd_1196_single_soma_neurons"
DATASET_PROOFREAD_AXON = "v1dd_1196_proofread_axons"
CAVE_REFERENCE_SPACE_ID = "v1dd_cave_nm"
STREAMLINE_REFERENCE_SPACE_ID = "v1dd_streamline_nm"
VOLUME_FEATURE_SET_ID = "v1dd_soma_volume_depth"
SOMA_FEATURE_SET_ID = "v1dd_soma_features"
HIERARCHY_ID = "v1dd_1196_cell_types"
RUN_ID = "v1dd_cell_type_annotation_r84"
ROOT_ID = "neuron"
SYNAPSE_TABLE_ID = "v1dd_1196_proofread_axon_to_single_soma"
SYNAPSE_FEATURE_MATRIX_ID = "v1dd_1196_proofread_axon_to_single_soma_features"
CONNECTOME_ID = "v1dd_1196_proofread_axon_to_single_soma"
QC_EMBEDDING_ID = "v1dd_r84_qc_nonnull_umap"
AD_REF_EMBEDDING_ID = "v1dd_r84_ad_ref_umap"
PUBLICATION = "https://github.com/AllenInstitute/v1dd_physiology"

SOMA_PATH = DATA_ROOT / "soma_and_cell_type_1196.feather"
R84_PATH = DATA_ROOT / "features_axon_dendrite_tables_v1dd_cell_type_axon_dendrite--r84.parquet"
AXON_PATH = DATA_ROOT / "proofread_axon_list_1196.npy"
# This list is single_somas joined to neurons_soma_model, despite its file name.
SINGLE_SOMA_PATH = DATA_ROOT / "proofread_dendrite_list_1196.npy"
SYNAPSE_PATH = DATA_ROOT / "syn_df_all_to_proofread_to_all_1196.feather"
SYNAPSE_LABEL_PATH = DATA_ROOT / "syn_label_df_all_to_proofread_to_all_1196.feather"
SOMA_FEATURES_PATH = DATA_ROOT / "soma_features.parquet"

required_paths = [
    SOMA_PATH, R84_PATH, AXON_PATH, SINGLE_SOMA_PATH, SYNAPSE_PATH,
    SYNAPSE_LABEL_PATH, SOMA_FEATURES_PATH, COLORS_PATH,
]
missing_paths = [path for path in required_paths if not path.is_file()]
assert not missing_paths, f"Missing local inputs: {missing_paths}"

## dataset dataitem

In [3]:
soma_source = pd.read_feather(SOMA_PATH)
r84_source = pd.read_parquet(R84_PATH)

# Cell-type annotations use nucleus ids; CCM DataItems use root ids so every
# downstream table refers to the reconstructed cell rather than the nucleus row.
cell_type_df = r84_source.rename(
    columns={"root_id": "r84_root_id", "cell_id": "id"}
)

soma_columns = {
    "id", "pt_root_id", "volume",
    "pt_position_x", "pt_position_y", "pt_position_z",
    "pt_position_trform_x", "pt_position_trform_y", "pt_position_trform_z",
}
# Only class and subclass annotations are retained from the cell-type table.
cell_type_columns = ["class", "subclass", "subclass_confidence"]
assert soma_columns <= set(soma_source.columns)
assert {"id", *cell_type_columns} <= set(cell_type_df.columns)
assert cell_type_df["id"].is_unique
assert set(cell_type_df["class"].dropna().unique()) == {"exc", "inh"}
assert cell_type_df["subclass"].notna().all()

soma_ct_df = soma_source[list(soma_columns)].merge(
    cell_type_df[["id"] + cell_type_columns],
    how="left",
    on="id",
    validate="many_to_one",
)

# DataItem identity must be unambiguous. Remove both one-nucleus-to-many-root and
# many-nuclei-to-one-root mappings instead of choosing an arbitrary record.
id_root_pairs = soma_ct_df[["id", "pt_root_id"]].drop_duplicates()
root_ids_per_id = id_root_pairs.groupby("id")["pt_root_id"].nunique()
ids_per_root_id = id_root_pairs.groupby("pt_root_id")["id"].nunique()
ambiguous_ids = root_ids_per_id.index[root_ids_per_id > 1]
ambiguous_root_ids = ids_per_root_id.index[ids_per_root_id > 1]
ambiguous_mask = (
    soma_ct_df["id"].isin(ambiguous_ids)
    | soma_ct_df["pt_root_id"].isin(ambiguous_root_ids)
)
soma_ct_unique_df = soma_ct_df.loc[~ambiguous_mask].reset_index(drop=True)
assert soma_ct_unique_df["id"].is_unique
assert soma_ct_unique_df["pt_root_id"].is_unique

# The release is scoped to single-soma neurons: segments holding exactly one soma
# that the neuron soma model also classifies as a neuron.
single_soma_source_ids = np.load(SINGLE_SOMA_PATH, allow_pickle=True)
soma_ct_unique_df = soma_ct_unique_df.loc[
    soma_ct_unique_df["pt_root_id"].isin(set(single_soma_source_ids))
].reset_index(drop=True)

# Dataset associations may only reference registered DataItems, so trim the axon
# cohort to roots retained by the identity cleanup.
unique_root_ids = np.array(sorted(set(soma_ct_unique_df["pt_root_id"])))
registered_ids = set(soma_ct_unique_df["pt_root_id"].astype(str))
axon_source_ids = np.load(AXON_PATH, allow_pickle=True)
axon_proof_root_ids = axon_source_ids[np.isin(axon_source_ids, unique_root_ids)]
print(
    f"single soma neurons: {len(soma_ct_unique_df):,} / "
    f"{len(single_soma_source_ids):,}"
)
print(f"proofread axons: {len(axon_proof_root_ids):,} / {len(axon_source_ids):,}")

single soma neurons: 63,986 / 63,986
proofread axons: 1,148 / 1,210


In [4]:
datasets = [
    DataSet(
        id=DATASET_SINGLE_SOMA_NEURONS,
        name="V1DD release 1196 - single soma neurons cohort",
        publication=PUBLICATION, modality=Modality.ELECTRON_MICROSCOPY,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_PROOFREAD_AXON, name="V1DD release 1196 - proofread axons cohort",
        publication=PUBLICATION, modality=Modality.ELECTRON_MICROSCOPY,
        project_id=PROJECT_ID,
    ),
]
print(write_models(datasets, output_root=OUTPUT_ROOT))

dataitems = [
    DataItem(id=str(row.pt_root_id), name=str(row.id), project_id=PROJECT_ID)
    for row in soma_ct_unique_df.itertuples()
]
print(write_models(dataitems, output_root=OUTPUT_ROOT))

associations = [
    DataItemDataSetAssociation(
        dataitem_id=str(root_id),
        dataset_id=DATASET_SINGLE_SOMA_NEURONS,
        project_id=PROJECT_ID,
    )
    for root_id in unique_root_ids
]
associations.extend(
    DataItemDataSetAssociation(
        dataitem_id=str(root_id), dataset_id=DATASET_PROOFREAD_AXON, project_id=PROJECT_ID
    )
    for root_id in axon_proof_root_ids
)
assert {row.dataitem_id for row in associations} <= registered_ids
print(write_models(associations, output_root=OUTPUT_ROOT))

WrittenResult(class_name='DataSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/dataset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=2)


WrittenResult(class_name='DataItem', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/dataitem'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=63986)


WrittenResult(class_name='DataItemDataSetAssociation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/dataitem_dataset_association'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataset_id" = source."dataset_id" AND target."dataitem_id" = source."dataitem_id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=65134)


## spatial location

In [5]:
default_view = Default2DView(
    left_to_right=SignedAxis.PLUS_X,
    bottom_to_top=SignedAxis.MINUS_Y,
)
cortical_orientation = AnatomicalOrientation(
    pia_to_white_matter=SignedAxis.PLUS_Y,
)
reference_spaces = [
    ReferenceSpace(
        id=CAVE_REFERENCE_SPACE_ID,
        name="V1DD CAVE nanometer coordinates",
        description="Soma nucleus positions from the local release table.",
        unit=Unit.NANOMETERS_LENGTH,
        default_2d_view=default_view,
        project_id=PROJECT_ID,
    ),
    ReferenceSpace(
        id=STREAMLINE_REFERENCE_SPACE_ID,
        name="V1DD streamline-transformed nanometer coordinates",
        description="Precomputed V1DD streamline-transformed soma positions.",
        unit=Unit.NANOMETERS_LENGTH,
        default_2d_view=default_view,
        anatomical_orientation=cortical_orientation,
        project_id=PROJECT_ID,
    ),
]
print(write_models(reference_spaces, output_root=OUTPUT_ROOT))

spatial_locations = []
for row in soma_ct_unique_df.itertuples():
    common = {
        "dataitem_id": str(row.pt_root_id),
        "location_type": LocationType.SOMA,
        "project_id": PROJECT_ID,
    }
    spatial_locations.extend([
        SpatialLocation(
            reference_space=CAVE_REFERENCE_SPACE_ID,
            description="Soma nucleus position from the local release table.",
            x=float(row.pt_position_x), y=float(row.pt_position_y), z=float(row.pt_position_z),
            **common,
        ),
        SpatialLocation(
            reference_space=STREAMLINE_REFERENCE_SPACE_ID,
            description="Streamline-transformed soma nucleus position.",
            x=float(row.pt_position_trform_x),
            y=float(row.pt_position_trform_y),
            z=float(row.pt_position_trform_z),
            **common,
        ),
    ])
print(write_models(spatial_locations, output_root=OUTPUT_ROOT))

WrittenResult(class_name='ReferenceSpace', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/referencespace'), mode='merge_scoped', predicates=('(target."project_id" IS NOT DISTINCT FROM source."project_id") AND target."id" = source."id"',), rows_written=2)


WrittenResult(class_name='SpatialLocation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/spatiallocation'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataitem_id" = source."dataitem_id" AND target."reference_space" = source."reference_space" AND target."location_type" = source."location_type" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=127972)


## features

The `v1dd_soma_volume_depth` feature set contains soma nucleus volume and soma depth. Depth is the precomputed streamline-transformed Y coordinate (`pt_position_trform_y`) in nanometers, matching the streamline spatial reference space.

In [6]:
volume_definition = CellFeatureDefinition(
    id="soma_volume",
    description="Nucleus volume from the local V1DD release table.",
    unit=Unit.MICRONS_CUBED.value,
    data_type="<f4",
    project_id=PROJECT_ID,
    feature_set_id=VOLUME_FEATURE_SET_ID,
)
depth_definition = CellFeatureDefinition(
    id="soma_depth",
    description="Soma depth from the streamline-transformed Y coordinate in the spatial location table.",
    unit=Unit.NANOMETERS_LENGTH.value,
    data_type="<f8",
    project_id=PROJECT_ID,
    feature_set_id=VOLUME_FEATURE_SET_ID,
)
volume_definitions = [volume_definition, depth_definition]
volume_feature_set = CellFeatureSet(
    id=VOLUME_FEATURE_SET_ID,
    description="V1DD release 1196 soma nucleus volume and streamline-transformed soma depth.",
    feature_definition_ids=[definition.id for definition in volume_definitions],
    extraction_method="Loaded from soma_and_cell_type_1196.feather.",
    project_id=PROJECT_ID,
)
print(write_models(volume_definitions, output_root=OUTPUT_ROOT))
print(write_models([volume_feature_set], output_root=OUTPUT_ROOT))

# Wide feature payloads use the DataItem id as their row key and carry project
# and feature-set scope columns alongside the measured values.
volume_wide = pd.DataFrame({
    "id": soma_ct_unique_df["pt_root_id"].astype(str),
    "soma_volume": soma_ct_unique_df["volume"].astype("float32"),
    "soma_depth": soma_ct_unique_df["pt_position_trform_y"].astype("float64"),
    "project_id": PROJECT_ID,
    "feature_set_id": VOLUME_FEATURE_SET_ID,
})
volume_schema = build_cell_feature_matrix_schema(
    volume_feature_set, volume_definitions, cell_index_column="id"
)
volume_table = pa.Table.from_pandas(
    volume_wide, schema=volume_schema, preserve_index=False
)
volume_path = (
    OUTPUT_ROOT / WIDE_PAYLOAD_PATHS["cell_features"] / VOLUME_FEATURE_SET_ID
)
write_deltalake(
    volume_path, volume_table, mode="overwrite", schema_mode="merge",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)

# CellFeatureMatrix is the CCM metadata pointer to the dataframe-backed payload.
volume_matrix = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{VOLUME_FEATURE_SET_ID}",
    feature_set_id=VOLUME_FEATURE_SET_ID,
    parquet_path=volume_path.relative_to(OUTPUT_ROOT).as_posix(),
    cell_index_column="id",
    project_id=PROJECT_ID,
)
print(write_models([volume_matrix], output_root=OUTPUT_ROOT))

WrittenResult(class_name='CellFeatureDefinition', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeaturedefinition'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."feature_set_id" = source."feature_set_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\') AND target."feature_set_id" IN (\'v1dd_soma_volume_depth\')',), rows_written=2)
WrittenResult(class_name='CellFeatureSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeatureset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=1)


WrittenResult(class_name='CellFeatureMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeaturematrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."feature_set_id" = source."feature_set_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=1)


## cell types

Assign class and subclass colors before writing the hierarchy so each run writes the final cluster records only once.

In [7]:
CLASS_CATEGORY = "Class"
SUBCLASS_CATEGORY = "Sub-class"
class_colors = {"exc": "#FF0000", "inh": "#0000FF"}


# Only cells with a complete class/subclass path enter the hierarchy.
ct_df = soma_ct_unique_df.dropna(subset=["class", "subclass"]).copy()
subclass_to_class_values = ct_df.groupby("subclass")["class"].unique()
assert all(len(values) == 1 for values in subclass_to_class_values)
subclass_to_class = {
    key: values[0] for key, values in subclass_to_class_values.items()
}
class_ids = sorted(ct_df["class"].unique())
subclass_ids = sorted(ct_df["subclass"].unique())

rtf_text = COLORS_PATH.read_text(encoding="cp1252")
palette = dict(re.findall(r'"([^"]+)"\s*:\s*"(#[0-9A-Fa-f]{6})"', rtf_text))
assert len(palette) == 14
# The palette is defined at subclass level; aliases bridge source labels to it.
source_to_palette = {"L5": "L5IT", "BipolarFamily": "Bipolarfamily"}

# CCM stores the explicit tree: root -> class -> subclass.
clusters = [Cluster(
    id=ROOT_ID, hierarchy_id=HIERARCHY_ID, children=class_ids, level=0
)]

for class_id in class_ids:
    children = sorted(
        subclass for subclass in subclass_ids
        if subclass_to_class[subclass] == class_id
    )

    clusters.append(Cluster(
        id=class_id, hierarchy_id=HIERARCHY_ID, parent=ROOT_ID,
        children=children, level=1, hierarchy_category=CLASS_CATEGORY,
        hex_color=class_colors[class_id],
    ))

# Subclasses are terminal nodes; the source cluster data is intentionally omitted.
for subclass in subclass_ids:
    clusters.append(Cluster(
        id=subclass, hierarchy_id=HIERARCHY_ID,
        parent=subclass_to_class[subclass],
        level=2, hierarchy_category=SUBCLASS_CATEGORY,
        hex_color=palette[source_to_palette.get(subclass, subclass)],
    ))

# Replace this hierarchy scope so obsolete leaf nodes cannot survive a rerun.
cluster_path = OUTPUT_ROOT / MODEL_TABLE_PATHS["Cluster"]
if DeltaTable.is_deltatable(str(cluster_path)):
    DeltaTable(str(cluster_path)).delete(f"hierarchy_id = '{HIERARCHY_ID}'")

print(write_models(clusters, output_root=OUTPUT_ROOT))
print(f"clusters: {len(clusters):,}")
print(f"colored clusters: {sum(row.hex_color is not None for row in clusters)} / {len(clusters)}")

WrittenResult(class_name='Cluster', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cluster'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'v1dd_1196_cell_types\')',), rows_written=16)
clusters: 16
colored clusters: 15 / 16


In [8]:
run = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="V1DD CSM cell-type annotation",
    algorithm_version="r84",
    input_dataset=DATASET_SINGLE_SOMA_NEURONS,
    json_object=json.dumps({
        "source": str(R84_PATH),
        "assignment_columns": ["class", "subclass"],
        "confidence_columns": ["subclass_confidence"],
    }),
)

categories = [
    HierarchyCategory(
        id=CLASS_CATEGORY, hierarchy_id=HIERARCHY_ID,
        description="Cell class from the r84 parquet class column.", level=1,
    ),

    HierarchyCategory(
        id=SUBCLASS_CATEGORY, hierarchy_id=HIERARCHY_ID,
        description="Cell sub-class from the r84 parquet subclass column.", level=2,
    ),
]

hierarchy = ClusterHierarchy(
    id=HIERARCHY_ID,
    run=RUN_ID,
    root=ROOT_ID,
    clusters=[cluster.id for cluster in clusters],
)


# Replace dependent scopes so obsolete categories and memberships cannot persist.
scoped_tables = {
    "HierarchyCategory": f"hierarchy_id = '{HIERARCHY_ID}'",
    "ClusterMembership": (
        f"project_id = '{PROJECT_ID}' AND hierarchy_id = '{HIERARCHY_ID}'"
    ),
}

for model_name, predicate in scoped_tables.items():
    table_path = OUTPUT_ROOT / MODEL_TABLE_PATHS[model_name]
    if DeltaTable.is_deltatable(str(table_path)):
        DeltaTable(str(table_path)).delete(predicate)


print(write_models([run], output_root=OUTPUT_ROOT))
print(write_models(categories, output_root=OUTPUT_ROOT))
print(write_models([hierarchy], output_root=OUTPUT_ROOT))

# Store each cell at the root, class, and terminal subclass levels so consumers
# can filter at any hierarchy level without recursively walking the tree.
memberships = []
for row in ct_df.to_dict("records"):
    assigned = [ROOT_ID, row["class"], row["subclass"]]
    memberships.extend(
        ClusterMembership(
            item=str(row["pt_root_id"]),
            cluster=cluster_id,
            hierarchy_id=HIERARCHY_ID,
            probability=1.0,
            project_id=PROJECT_ID,
        )
        for cluster_id in assigned
    )

print(write_models(memberships, output_root=OUTPUT_ROOT))


# Retained because the membership batch is freed before the verification cell.
membership_count = len(memberships)
print(f"annotated cells: {len(ct_df):,} | memberships: {membership_count:,}")

WrittenResult(class_name='AlgorithmRun', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/algorithmrun'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=1)
WrittenResult(class_name='HierarchyCategory', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/hierarchycategory'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'v1dd_1196_cell_types\')',), rows_written=2)
WrittenResult(class_name='ClusterHierarchy', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/clusterhierarchy'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=1)


WrittenResult(class_name='ClusterMembership', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/clustermembership'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."hierarchy_id" = source."hierarchy_id" AND target."item" = source."item" AND target."cluster" = source."cluster" AND target."project_id" IN (\'v1dd_1196\') AND target."hierarchy_id" IN (\'v1dd_1196_cell_types\')',), rows_written=173370)
annotated cells: 57,790 | memberships: 173,370


## soma features

In [9]:
soma_features = pl.read_parquet(SOMA_FEATURES_PATH)

# Match measurements by neuron/nucleus id and retain the CCM DataItem root id.
soma_feature_ids = pl.from_pandas(soma_ct_unique_df[["id", "pt_root_id"]]).select(
    pl.col("id").alias("nucleus_id"),
    pl.col("pt_root_id").cast(pl.String).alias("id"),
)
soma_features_cc = (
    soma_feature_ids.join(soma_features, on="nucleus_id", how="inner", validate="1:1")
)
assert set(soma_features_cc["id"].to_list()) == registered_ids
feature_columns = [
    column for column in soma_features_cc.columns
    if column not in {"soma_id", "nucleus_id", "id"}
]
unit_by_feature = {
    "nucleus_volume_um": Unit.MICRONS_CUBED.value,
    "nucleus_area_um": Unit.MICRONS_SQUARE.value,
    "nuclear_area_to_volume_ratio": Unit.RATIO.value,
    "nuclear_folding_area_um": Unit.MICRONS_SQUARE.value,
    "fraction_nuclear_folding": Unit.RATIO.value,
    "nucleus_to_soma_ratio": Unit.RATIO.value,
    "soma_volume_um": Unit.MICRONS_CUBED.value,
    "soma_area_um": Unit.MICRONS_SQUARE.value,
    "soma_to_nucleus_center_dist": Unit.MICRONS_LENGTH.value,
    "soma_area_to_volume_ratio": Unit.RATIO.value,
    "soma_synapse_density_um": Unit.COUNT_PER_MICRONS_CUBED.value,
}
assert set(feature_columns) == set(unit_by_feature)
feature_definitions = [
    CellFeatureDefinition(
        id=column, description=column.replace("_", " "),
        unit=unit_by_feature[column], data_type="<f8",
        project_id=PROJECT_ID, feature_set_id=SOMA_FEATURE_SET_ID,
    )
    for column in feature_columns
]
soma_feature_set = CellFeatureSet(
    id=SOMA_FEATURE_SET_ID,
    description="V1DD soma morphology and nucleus-derived features.",
    feature_definition_ids=[definition.id for definition in feature_definitions],
    extraction_method="Loaded from soma_features.parquet.",
    project_id=PROJECT_ID,
)
print(write_models(feature_definitions, output_root=OUTPUT_ROOT))
print(write_models([soma_feature_set], output_root=OUTPUT_ROOT))

# Store measurements in a typed wide table and register its location separately.
soma_wide = (
    soma_features_cc.select(["id"] + feature_columns)
    .with_columns([pl.col(column).cast(pl.Float64) for column in feature_columns])
    .to_pandas()
)
soma_wide["project_id"] = PROJECT_ID
soma_wide["feature_set_id"] = SOMA_FEATURE_SET_ID
soma_schema = build_cell_feature_matrix_schema(
    soma_feature_set, feature_definitions, cell_index_column="id"
)
soma_table = pa.Table.from_pandas(
    soma_wide, schema=soma_schema, preserve_index=False
)
soma_feature_path = (
    OUTPUT_ROOT / WIDE_PAYLOAD_PATHS["cell_features"] / SOMA_FEATURE_SET_ID
)
write_deltalake(
    soma_feature_path, soma_table, mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
soma_matrix = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{SOMA_FEATURE_SET_ID}",
    feature_set_id=SOMA_FEATURE_SET_ID,
    parquet_path=soma_feature_path.relative_to(OUTPUT_ROOT).as_posix(),
    cell_index_column="id", project_id=PROJECT_ID,
)
print(write_models([soma_matrix], output_root=OUTPUT_ROOT))
print(f"soma feature rows: {soma_table.num_rows:,}")

WrittenResult(class_name='CellFeatureDefinition', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeaturedefinition'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."feature_set_id" = source."feature_set_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\') AND target."feature_set_id" IN (\'v1dd_soma_features\')',), rows_written=11)
WrittenResult(class_name='CellFeatureSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeatureset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=1)


WrittenResult(class_name='CellFeatureMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeaturematrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."feature_set_id" = source."feature_set_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=1)
soma feature rows: 63,986


## embeddings

In [10]:
embedding_columns = {
    QC_EMBEDDING_ID: ("qc_nonnull_embed0", "qc_nonnull_embed1"),
    AD_REF_EMBEDDING_ID: ("ad_ref_embed0", "ad_ref_embed1"),
}
embedding_spaces = [
    EmbeddingSpace(
        id=QC_EMBEDDING_ID,
        name="V1DD r84 QC-nonnull UMAP",
        description="QC-nonnull UMAP coordinates from the r84 cell-type parquet. Covers more cells but uses less features so less accurate",
        embedding_method=EmbeddingMethod.UMAP,
        parameters_json=json.dumps(
            {"source_columns": embedding_columns[QC_EMBEDDING_ID]}
        ),
        input_features_description=(
            "QC-nonnull axon/dendrite features supplied in the r84 parquet."
        ),
        project_id=PROJECT_ID,
    ),
    EmbeddingSpace(
        id=AD_REF_EMBEDDING_ID,
        name="V1DD r84 axon-dendrite reference UMAP",
        description=(
            "Axon-dendrite reference UMAP coordinates from the r84 cell-type parquet. Covers about half of the cells but separating better than the QC-nonnull UMAP."
        ),
        embedding_method=EmbeddingMethod.UMAP,
        parameters_json=json.dumps(
            {"source_columns": embedding_columns[AD_REF_EMBEDDING_ID]}
        ),
        input_features_description=(
            "Reduced axon-dendrite reference features supplied in the r84 parquet."
        ),
        project_id=PROJECT_ID,
    ),
]
print(write_models(embedding_spaces, output_root=OUTPUT_ROOT))

# Embedding rows use nucleus ids, so translate them to the canonical CCM
# DataItem ids before creating locations.
dataitem_lookup = pl.DataFrame({
    "nucleus_id": soma_ct_unique_df["id"].astype(str),
    "dataitem_id": soma_ct_unique_df["pt_root_id"].astype(str),
})
embedding_source = (
    pl.read_parquet(R84_PATH)
    .with_columns(pl.col("cell_id").cast(pl.String).alias("nucleus_id"))
    .join(dataitem_lookup, on="nucleus_id", how="inner")
    .unique(subset=["dataitem_id"], keep="first")
)
locations_by_space = {}
for embedding_space, (x_column, y_column) in embedding_columns.items():
    # Missing coordinates are space-specific; retain every finite point per space.
    coordinates = embedding_source.filter(
        pl.col(x_column).is_finite() & pl.col(y_column).is_finite()
    )
    locations = [
        EmbeddingLocation(
            dataitem_id=row["dataitem_id"],
            embedding_space=embedding_space,
            x=float(row[x_column]),
            y=float(row[y_column]),
            project_id=PROJECT_ID,
        )
        for row in coordinates.iter_rows(named=True)
    ]
    locations_by_space[embedding_space] = locations
    print(write_models(locations, output_root=OUTPUT_ROOT))
    print(f"{embedding_space}: {len(locations):,}")

WrittenResult(class_name='EmbeddingSpace', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/embeddingspace'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=2)


WrittenResult(class_name='EmbeddingLocation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/embeddinglocation'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataitem_id" = source."dataitem_id" AND target."embedding_space" = source."embedding_space" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=57790)
v1dd_r84_qc_nonnull_umap: 57,790


WrittenResult(class_name='EmbeddingLocation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/embeddinglocation'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataitem_id" = source."dataitem_id" AND target."embedding_space" = source."embedding_space" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=39114)
v1dd_r84_ad_ref_umap: 39,114


## synapses connectivity

Only proofread axon to registered single-soma synapses are saved, with matching
feature payloads and cell-cell connectivity.

In [11]:
# Release completed model batches before loading the multi-million-row synapse
# table; CCM bulk tables are written column-wise instead of as Pydantic objects.
del spatial_locations, associations, dataitems, memberships, volume_wide, volume_table
gc.collect()

synapse_source = feather.read_table(SYNAPSE_PATH, memory_map=False)
synapse_source = synapse_source.filter(
    pc.and_(
        pc.is_in(
            synapse_source["pre_pt_root_id"],
            value_set=pa.array(
                [int(value) for value in axon_proof_root_ids],
                type=synapse_source["pre_pt_root_id"].type,
            ),
        ),
        pc.is_in(
            synapse_source["post_pt_root_id"],
            value_set=pa.array(
                [int(value) for value in unique_root_ids],
                type=synapse_source["post_pt_root_id"].type,
            ),
        ),
    )
)
synapse_count = synapse_source.num_rows
assert pc.count_distinct(synapse_source["id"]).as_py() == synapse_count
print(f"proofread axon to single-soma synapses: {synapse_count:,}")

# SynapseConnectivityLong contains only connection identity and table scope.
# Per-synapse measurements are stored separately in the wide feature payload.
long_schema = build_arrow_schema(SynapseConnectivityLong)
long_table = pa.Table.from_arrays(
    [
        pa.repeat(PROJECT_ID, synapse_count),
        pc.cast(synapse_source["id"], pa.string()),
        pc.cast(synapse_source["pre_pt_root_id"], pa.string()),
        pc.cast(synapse_source["post_pt_root_id"], pa.string()),
        pa.repeat(SYNAPSE_TABLE_ID, synapse_count),
    ],
    schema=long_schema,
)
long_table = attach_linkml_metadata(
    long_table,
    linkml_class="SynapseConnectivityLong",
)
write_deltalake(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["SynapseConnectivityLong"],
    long_table,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"synapse_table_id = '{SYNAPSE_TABLE_ID}'"
    ),
    partition_by=["project_id"],
)
del long_table
gc.collect()

# Root ids define connectivity and are excluded from the feature matrix. The
# label Feather stores synapse ids in its index, exposed here as the id column.
feature_columns = [
    column
    for column in synapse_source.column_names
    if column != "id" and not column.endswith("_root_id")
]
feature_table = synapse_source.select(["id"] + feature_columns).set_column(
    0,
    "id",
    pc.cast(synapse_source["id"], pa.string()),
)
label_table = feather.read_table(
    SYNAPSE_LABEL_PATH,
    memory_map=False,
).select(["id", "tag"])
label_table = label_table.set_column(
    0,
    "id",
    pc.cast(label_table["id"], pa.string()),
).rename_columns(["id", "synaptictargetlabel"])
label_table = label_table.filter(
    pc.is_in(label_table["id"], value_set=feature_table["id"])
)
assert pc.count_distinct(label_table["id"]).as_py() == label_table.num_rows

# Left join keeps every selected synapse; unlabeled targets remain null.
synapse_wide = feature_table.join(
    label_table,
    keys="id",
    join_type="left outer",
)
assert synapse_wide.num_rows == synapse_count
label_count = pc.count(synapse_wide["synaptictargetlabel"]).as_py()
synapse_wide = synapse_wide.append_column(
    "project_id",
    pa.repeat(PROJECT_ID, synapse_count),
)
synapse_wide = synapse_wide.append_column(
    "synapse_table_id",
    pa.repeat(SYNAPSE_TABLE_ID, synapse_count),
)
synapse_feature_path = (
    OUTPUT_ROOT
    / WIDE_PAYLOAD_PATHS["synapse_features"]
    / SYNAPSE_FEATURE_MATRIX_ID
)
write_deltalake(
    synapse_feature_path,
    synapse_wide,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"synapse_table_id = '{SYNAPSE_TABLE_ID}'"
    ),
    partition_by=["project_id"],
)
synapse_feature_matrix = SynapseFeatureMatrix(
    id=SYNAPSE_FEATURE_MATRIX_ID,
    description=(
        "Per-synapse position, size and morphology-based target label for V1DD 1196 "
        "proofread axon to single-soma synapses."
    ),
    synapse_table_id=SYNAPSE_TABLE_ID,
    project_id=PROJECT_ID,
    parquet_path=synapse_feature_path.relative_to(OUTPUT_ROOT).as_posix(),
    synapse_index_column="id",
)
print(write_models([synapse_feature_matrix], output_root=OUTPUT_ROOT))
print(f"synapses: {synapse_count:,} | labelled: {label_count:,}")
del synapse_source, feature_table, label_table, synapse_wide
gc.collect()

proofread axon to single-soma synapses: 1,427,418


WrittenResult(class_name='SynapseFeatureMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/synapsefeaturematrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'v1dd_1196\')',), rows_written=1)
synapses: 1,427,418 | labelled: 1,403,249


20

In [12]:
# Read projected Feather batches and retain numeric IDs until pair aggregation.
synapse_columns = ["id", "pre_pt_root_id", "post_pt_root_id", "size"]
with pa.OSFile(str(SYNAPSE_PATH), "rb") as source:
    source_schema = pa.ipc.open_file(source).schema
    source_reader = pa.ipc.open_file(
        source,
        options=pa.ipc.IpcReadOptions(included_fields=[
            source_schema.get_field_index(column) for column in synapse_columns
        ]),
    )
    selected_batches = []
    for index in range(source_reader.num_record_batches):
        batch = source_reader.get_batch(index)
        selected_batches.append(batch.filter(pc.and_(
            pc.is_in(batch["pre_pt_root_id"], value_set=pa.array(axon_proof_root_ids)),
            pc.is_in(batch["post_pt_root_id"], value_set=pa.array(unique_root_ids)),
        )))
    selected_synapses = pl.from_arrow(
        pa.Table.from_batches(selected_batches, schema=source_reader.schema)
    ).rename({
        "pre_pt_root_id": "presynaptic_cell",
        "post_pt_root_id": "postsynaptic_cell",
    }).with_columns(
        pl.lit(PROJECT_ID).alias("project_id"),
        pl.lit(SYNAPSE_TABLE_ID).alias("synapse_table_id"),
    )
del selected_batches, source_reader
assert selected_synapses.height == synapse_count
selected_size_sum = selected_synapses["size"].sum()
pair_count = selected_synapses.select(
    "presynaptic_cell", "postsynaptic_cell",
).n_unique()

# Expand pair aggregates in bounded batches, committing the full scope atomically.
connectivity_schema = attach_linkml_metadata(
    pa.Table.from_batches([], schema=build_arrow_schema(CellCellConnectivityLong)),
    linkml_class="CellCellConnectivityLong",
).schema
connectivity_batches = iter_cell_cell_connectivity(
    selected_synapses,
    project_id=PROJECT_ID,
    connectome_id=CONNECTOME_ID,
    modality=Modality.ELECTRON_MICROSCOPY,
    size_column="size",
    size_unit=Unit.ARBITRARY_UNIT,
    batch_size=65_536,
)
connectivity_reader = pa.RecordBatchReader.from_batches(
    connectivity_schema,
    (
        batch
        for frame in connectivity_batches
        for batch in cell_cell_connectivity_to_arrow(frame).to_batches()
    ),
)
write_deltalake(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["CellCellConnectivityLong"],
    connectivity_reader,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"connectome_id = '{CONNECTOME_ID}'"
    ),
    partition_by=["project_id", "connectome_id", "measurement_type"],
    target_file_size=32 * 1024 * 1024,
)
print(
    f"connectivity: {pair_count:,} pairs | "
    f"{pair_count * 2:,} measurements"
)
del selected_synapses, connectivity_batches, connectivity_reader
gc.collect()

connectivity: 735,973 pairs | 1,471,946 measurements


10

## verify

In [13]:
# Read each CCM model table within this project's scope so unrelated projects
# sharing the output root do not affect the checks.
stored_datasets = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["DataSet"]).filter(
    pl.col("project_id") == PROJECT_ID
)
stored_items = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["DataItem"]).filter(
    pl.col("project_id") == PROJECT_ID
)
stored_associations = pl.read_delta(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["DataItemDataSetAssociation"]
).filter(pl.col("project_id") == PROJECT_ID)
stored_locations = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["SpatialLocation"]).filter(
    pl.col("project_id") == PROJECT_ID
)
stored_clusters = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["Cluster"]).filter(
    pl.col("hierarchy_id") == HIERARCHY_ID
)
stored_memberships = pl.read_delta(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["ClusterMembership"]
).filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("hierarchy_id") == HIERARCHY_ID)
)

# Confirm expected row counts, unique spatial keys, and valid references between
# datasets, DataItems, hierarchy nodes, and memberships.
assert set(stored_datasets["id"]) == {
    DATASET_SINGLE_SOMA_NEURONS,
    DATASET_PROOFREAD_AXON,
}
assert stored_items.height == len(soma_ct_unique_df)
assert set(stored_items["id"]) == registered_ids
assert stored_locations.height == 2 * len(soma_ct_unique_df)
assert stored_locations.select(
    "dataitem_id",
    "reference_space",
    "location_type",
).unique().height == stored_locations.height
association_counts = dict(
    stored_associations.group_by("dataset_id").len().iter_rows()
)
assert association_counts == {
    DATASET_SINGLE_SOMA_NEURONS: len(unique_root_ids),
    DATASET_PROOFREAD_AXON: len(axon_proof_root_ids),
}
assert stored_clusters.height == len(clusters)
assert stored_clusters.filter(pl.col("hex_color").is_not_null()).height == sum(
    cluster.hex_color is not None for cluster in clusters
)
assert stored_memberships.height == membership_count
assert set(stored_memberships["item"]) <= registered_ids
assert set(stored_memberships["cluster"]) <= set(stored_clusters["id"])

# Wide feature payloads should contain one stored row per retained DataItem in
# the population used to build each matrix.
stored_volume = pl.read_delta(volume_path).filter(pl.col("project_id") == PROJECT_ID)
stored_soma_features = pl.read_delta(soma_feature_path).filter(
    pl.col("project_id") == PROJECT_ID
)
assert stored_volume.height == len(soma_ct_unique_df)
expected_volume_features = pd.DataFrame({
    "id": soma_ct_unique_df["pt_root_id"].astype(str),
    "soma_volume": soma_ct_unique_df["volume"].astype("float32"),
    "soma_depth": soma_ct_unique_df["pt_position_trform_y"].astype("float64"),
}).set_index("id").sort_index()
pd.testing.assert_frame_equal(
    stored_volume.select("id", "soma_volume", "soma_depth")
    .to_pandas().set_index("id").sort_index(),
    expected_volume_features,
    check_exact=True,
)
assert stored_soma_features.height == soma_features_cc.height
assert set(stored_soma_features["id"]) == registered_ids

# Stream counts for large Delta tables so verification does not materialize
# millions of rows in memory.
stored_synapse_count = (
    pl.scan_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["SynapseConnectivityLong"])
    .filter(
        (pl.col("project_id") == PROJECT_ID)
        & (pl.col("synapse_table_id") == SYNAPSE_TABLE_ID)
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)
stored_synapse_feature_count = (
    pl.scan_delta(synapse_feature_path)
    .filter(
        (pl.col("project_id") == PROJECT_ID)
        & (pl.col("synapse_table_id") == SYNAPSE_TABLE_ID)
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

# Each connected pair should have the two derived measurement types, while the
# long synapse table and its feature matrix remain row-aligned.
connectivity_checks = (
    pl.scan_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["CellCellConnectivityLong"])
    .filter(
        (pl.col("project_id") == PROJECT_ID)
        & (pl.col("connectome_id") == CONNECTOME_ID)
        & (pl.col("synapse_table_id") == SYNAPSE_TABLE_ID)
    )
    .select(
        pl.len().alias("rows"),
        pl.col("measurement_type").n_unique().alias("measurement_types"),
        pl.col("presynaptic_cell").is_in(list(map(str, axon_proof_root_ids))).all().alias("valid_pre"),
        pl.col("postsynaptic_cell").is_in(list(registered_ids)).all().alias("valid_post"),
        pl.col("value").filter(pl.col("measurement_type") == "SYNAPSE_COUNT").sum().alias("count"),
        pl.col("value").filter(pl.col("measurement_type") == "SUM_ANATOMICAL_SIZE").sum().alias("size"),
    )
    .collect(engine="streaming")
    .row(0, named=True)
)
assert stored_synapse_count == synapse_count
assert stored_synapse_feature_count == synapse_count
assert connectivity_checks["rows"] == pair_count * 2
assert connectivity_checks["measurement_types"] == 2
assert connectivity_checks["valid_pre"] and connectivity_checks["valid_post"]
assert connectivity_checks["count"] == synapse_count
assert connectivity_checks["size"] == selected_size_sum

# Every embedding location must belong to a registered DataItem and match the
# number of finite source coordinates written for that embedding space.
for embedding_space, locations in locations_by_space.items():
    stored = read_embedding_locations(
        PROJECT_ID,
        embedding_space_ids=embedding_space,
        output_root=OUTPUT_ROOT,
    )
    assert stored.height == len(locations)
    assert set(stored["dataitem_id"]) <= registered_ids
print(
    "V1DD 1196 datasets, spatial data, features, taxonomy, "
    "synapses, connectivity, colors, and embeddings verified."
)

V1DD 1196 datasets, spatial data, features, taxonomy, synapses, connectivity, colors, and embeddings verified.
